## Información
### Objetivo
Este notebook implementa la primera etapa del pipeline RAG: la adquisición y preparación de la base documental. Su propósito es obtener automáticamente los documentos oficiales de la Municipalidad de Valdivia, extraer su contenido y realizar un proceso de limpieza y normalización que permita su posterior procesamiento.
### Objetivos específicos
* Identificar las fuentes documentales del sistema.
* Realizar la extracción automática de documentos mediante scraping.
* Descargar archivos en distintos formatos (PDF, HTML y DOCX).
* Extraer el contenido textual preservando la mayor cantidad de información posible.
* Eliminar elementos irrelevantes (encabezados, pies de página, espacios redundantes, caracteres especiales, etc.).
* Normalizar el texto para facilitar las etapas posteriores del pipeline.
* Almacenar los documentos limpios para su utilización en el proceso de chunking.
### Entradas
* Sitio web DOM Digital de la Municipalidad de Valdivia.
* Portal de Transparencia Municipal.
* Documentos oficiales en formato PDF, HTML y DOCX.
### Salidas
* Documentos descargados en data/raw/.
* Texto limpio y normalizado almacenado en data/processed/.
* Registro de la extracción y del procesamiento realizado.

# 01 Data Ingestion
---

### Contexto dentro del pipeline RAG

Hasta esta etapa el flujo del proyecto es:
```
PDF
↓
Document
↓
Ingesta y preparación de documentos  ← Etapa actual
↓
Chunking
↓
Embeddings 
```
En la siguiente etapa estos embeddings serán almacenados en una base vectorial para recuperar la información más relevante ante una consulta del usuario.

---
### Parte 1. Document Ingestion

In [1]:
import sys
from pathlib import Path


# Add project root to path
root = Path.cwd().resolve()
if not (root / "src").exists():
    for parent in root.parents:
        if (parent / "src").exists():
            root = parent
            break
sys.path.insert(0, str(root))

from src.ingestion.scraper import download_pdf

output_dir = Path("../data/raw/muni_valdivia/planes/")


url = (
    "https://munivaldivia.cl/wp-content/uploads/2025/11/"
    "Plan-Comunal-de-Emergencia-2025-2.pdf"
)

pdf_path = download_pdf(
    url=url,
    output_dir=output_dir

)

pdf_path.stat().st_size

print(f"Archivo descargado en: {pdf_path}")	
print(f"Tamaño del archivo: {pdf_path.stat().st_size} bytes")

Archivo descargado en: ..\data\raw\muni_valdivia\planes\Plan-Comunal-de-Emergencia-2025-2.pdf
Tamaño del archivo: 1256600 bytes


In [2]:
import src.ingestion.loaders

document = src.ingestion.loaders.load_pdf(pdf_path)

print("id             :", document.id)
print("Título         :", document.title)
print("Tipo de archivo:", document.file_type)
print("Páginas        :", document.metadata.get("pages", "N/A"))
print("--------------------------------------------")
print("Contenido", document.content[:150])


id             : 29f14155-d17b-4676-8a1b-8bc12f22d47d
Título         : Plan-Comunal-de-Emergencia-2025-2
Tipo de archivo: pdf
Páginas        : 33
--------------------------------------------
Contenido  
Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página  
1 de 33 
Fecha: Agosto-20


---
## Parte 2. Document Cleaning

In [3]:
from src.processing.cleaner import clean_document

cleaned_document = clean_document(document)	

print(cleaned_document.content[:150])

Ilustre Municipalidad de Valdivia PLANTILLA 
VERSION: 01 
PLAN COMUNAL DE EMERGENCIA – ANEXOS PLANES POR AMENAZA 
Página 
1 de 33 
Fecha: Agosto-2025 


In [4]:
print("Documento original : ", len(document.content), "caracteres.")
print("Documento limpio   : ", len(cleaned_document.content), "caracteres.")

Documento original :  90760 caracteres.
Documento limpio   :  90189 caracteres.


---
## Parte 3. Exportar documento 	 
Exportar a data/processed/ para su posterior uso en la etapa de Chunking.

In [5]:
# EXPORTAR "cleaned_document" A data/processed/  en formato json

import json
from pathlib import Path

# Crear la carpeta data/processed si no existe
processed_dir = Path("../data/processed/")
processed_dir.mkdir(parents=True, exist_ok=True)

# Definir la ruta del archivo JSON
json_path = processed_dir / f"{cleaned_document.title}.json"

# Exportar el documento a JSON
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump({
        'id': cleaned_document.id,
        'title': cleaned_document.title,
        'source': str(cleaned_document.source),
        'file_type': cleaned_document.file_type,
        'content': cleaned_document.content,
        'metadata': cleaned_document.metadata
    }, f, ensure_ascii=False, indent=4)

print(f"Documento exportado a: {json_path}")

Documento exportado a: ..\data\processed\Plan-Comunal-de-Emergencia-2025-2.json


---
## ✅ Conclusiones

En esta sección aprendimos que:

- Un documento puede descargarse desde una URL oficial.
- El loader transforma el PDF en un objeto Document.
- El cleaner prepara el texto para las siguientes etapas.
- Cada módulo tiene una única responsabilidad dentro del pipeline.